# Chapter 4: Tokenization

[Read this chapter online](https://jackluu.io/book/section-1-foundations/ch04-tokenization/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch04-tokenization.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 4: Tokenization

![Where we are](../assets/diagrams/ch04-where-we-are.png){ width="756" }
*Figure 4.1: Tokenization in the big picture.*

Now that we have tensors to hold our data from the previous chapter, we are ready to move to the next stage in our map (Figure 4.1). Before a language model can find patterns in text, we must translate that text into a format it can understand. Neural networks only do math, which means they only eat numbers. In this chapter, we build our tokenizer: a translator that chops text into small pieces and assigns a number to each one.

In this chapter you will:

- Understand why neural networks need numbers
- Build a character-level vocabulary from the Shakespeare dataset
- Write functions to encode text into integers and decode it back
- Prepare the data for training by splitting it into two sets

**Words to Know**
    - **Token**: one small piece of text, here a single character.
    - **Tokenizer**: a function that turns text into tokens and then into numbers.

## Theory

### Neural Networks Are Number Machines

![Text goes in, numbers come out: each character maps to one ID](../assets/diagrams/ch04-text-to-ids.png){ width="398" }
*Figure 4.2: A tokenizer turns each character into a number.*

Neural networks cannot read. You cannot feed them a string like "Hello". They expect a grid of numbers to multiply and add. To feed text into a model, we must first convert it into numbers. This process, illustrated in Figure 4.2, is called tokenization.

We need a consistent rulebook. If the letter `a` is the number `0`, it must always be `0`. We call this rulebook our **vocabulary**.

### The Simplest Tokenizer

There are many ways to tokenize text. You could map each full word to a number (word-level). You could group common letters like `th` together (subword-level). 

For our model, we use the simplest approach: **character-level tokenization**.
Every unique character in our dataset gets its own unique integer.
If our vocabulary is just `{a: 0, b: 1, c: 2}`, then the word "cab" becomes `[2, 0, 1]`.

The Shakespeare dataset contains exactly 65 unique characters. This includes uppercase letters, lowercase letters, spaces, and punctuation. If we give each one an ID from 0 to 64, we can translate any Shakespearean sentence into a list of integers.

**In Business**
    When you build an assistant to draft text in your company's house style, tokenization choices matter. A character-level tokenizer is simple but makes sequences long. A word-level tokenizer makes sequences short but requires a massive vocabulary. Modern business assistants use Byte-Pair Encoding (BPE), a middle ground that groups common character sequences into single tokens. We use character-level here because it keeps the math clean while our model learns from its archive (our Shakespeare dataset).

### The Training and Validation Split

![Splitting the dataset into training and validation sets](../assets/diagrams/ch04-train-val-split.png){ width="418" }
*Figure 4.3: Hiding part of the data lets us test if the model actually learned.*

Once we encode all of Shakespeare into one massive list of numbers, we split it into two piles (Figure 4.3):

- **Training set** (90%): The data the model studies to learn patterns.
- **Validation set** (10%): The data we hide from the model, used to test it later.

Why hide data? We want to know if the model is picking up patterns that hold across the text, or just memorizing the passages it was shown. If it performs well on the training set but fails on the validation set, it is overfitting.

**Watch Out**
    Our tokenizer only knows the characters it saw in the training data. If you feed it a digit (like `1` or `2`), an emoji, or a Chinese character, Python will raise a `KeyError` because that character is not in our 65-character vocabulary.

## Code

We write our tokenizer in Python, load the dataset, and encode it.

> **File**: `src/ch03_tokenizer.py`
> **Run it**: `python src/ch03_tokenizer.py`

```python
def build_vocab(text):
    # Find all unique characters in the text
    chars = sorted(set(text))
    
    # Dictionaries to translate between characters and their integer IDs
    char_to_id = {ch: i for i, ch in enumerate(chars)}
    id_to_char = {i: ch for i, ch in enumerate(chars)}
    
    return chars, char_to_id, id_to_char
```

Here is what this code does:

1. Line 3 uses `set(text)` to find every unique character in the entire text.
2. Line 3 uses `sorted()` to put them in alphabetical order so the IDs stay consistent.
3. Lines 6 and 7 build two dictionaries: one to go from character to ID (`char_to_id`), and one to go back (`id_to_char`).

![Code flow for the tokenizer script](../assets/diagrams/ch04-code-flow.png){ width="318" }
*Figure 4.4: How the tokenizer processes the dataset.*

As Figure 4.4 shows, the script processes the dataset step by step. Next, we write the translation functions:

```python
def encode(text, char_to_id):
    # Convert a string into a list of integer IDs
    return [char_to_id[ch] for ch in text]

def decode(ids, id_to_char):
    # Convert a list of integer IDs back into a string
    return "".join([id_to_char[i] for i in ids])
```

Line 3 converts a string into a list of integer IDs, and line 7 converts them back into a string.

Let's run the script and see what our 65 characters look like, and test a quick round-trip.

```python
$ python src/ch03_tokenizer.py
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
...
--- Testing encode / decode ---
Original : 'Hello, World!'
Encoded  : [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Decoded  : 'Hello, World!'
Round-trip matches: True

--- Encoding the full dataset ---
```

**What just happened:**

- The script loaded over a million characters from the dataset.
- It found exactly 65 unique characters. Notice the space and newline characters at the start!
- It translated the string "Hello, World!" into a list of integers.
- It translated those integers back to prove no data was lost.

Finally, we encode the entire dataset into a PyTorch tensor and split it.

```python
Training tokens  : 1,003,854
Validation tokens: 111,540

Tokenizer ready! Ready for Chapter 5.
```

## Try It

See how different inputs are converted into numbers. We wrote a short script that imports our tokenizer and encodes a few business phrases.

**Try It**
    Open the terminal and run the example script. Notice how every letter, space, and punctuation mark is assigned a specific number from our vocabulary.

    ```console title="Terminal"
    $ python src/examples/ch04_tokenizer_demo.py
    Demo: Encoding short business phrases
    'invoice'       -> [47, 52, 60, 53, 47, 41, 43]
    'URGENT!'       -> [33, 30, 19, 17, 26, 32, 2]
    'Hello.'        -> [20, 43, 50, 50, 53, 8]

    ```

## Key Takeaways

- Neural networks process numbers, not text.
- Tokenization translates text into numbers using a fixed vocabulary.
- We use a character-level tokenizer with a vocabulary of 65 characters.
- `encode()` turns strings into lists of IDs; `decode()` turns IDs back into strings.
- We split our data into a training set (to learn) and a validation set (to test).

## Check Your Understanding

1. Why must we convert text to numbers before feeding it to a language model?
2. In our character-level tokenizer, what happens if we try to encode a character that wasn't in the training data?
3. What is the purpose of the validation set?
4. How many unique characters are in our vocabulary?


## Further Reading

**Why real models do not use a character vocabulary.** Chapter 4 gives every character its own token, which keeps the vocabulary tiny and the code short. Production models split text into word pieces instead: common words stay whole, rare ones break into parts, and nothing is ever unknown. This paper is the method, and it is still the basis of the tokenizers shipped with today's models.

<div class="refs" markdown>

Sennrich, R., Haddow, B., & Birch, A. (2015). *Neural machine translation of rare words with subword units* (arXiv:1508.07909). arXiv. https://doi.org/10.48550/arXiv.1508.07909

</div>

---

### `src/ch03_tokenizer.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch03_tokenizer.py"   # a cell has none, and the file uses it to find the text

"""
Build a character-level tokenizer.
This file belongs to Chapter 4.
Run: python src/ch03_tokenizer.py
"""
import os
import sys
import torch

# Settings
DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

# --- The Idea ---

def build_vocab(text):
    # Find all unique characters in the text
    chars = sorted(set(text))

    # Dictionaries to translate between characters and their integer IDs
    char_to_id = {ch: i for i, ch in enumerate(chars)}
    id_to_char = {i: ch for i, ch in enumerate(chars)}

    return chars, char_to_id, id_to_char

def encode(text, char_to_id):
    # Convert a string into a list of integer IDs
    return [char_to_id[ch] for ch in text]

def decode(ids, id_to_char):
    # Convert a list of integer IDs back into a string
    return "".join([id_to_char[i] for i in ids])

# --- Demo ---
if __name__ == "__main__":
    print("Chapter 4: Tokenization\n")

    if not os.path.exists(DATA_PATH):
        print("ERROR: shakespeare.txt not found.")
        print("Please run:  python src/utils/download_data.py")
        sys.exit(1)

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"Loaded {len(text):,} characters from shakespeare.txt")
    print(f"First 100 chars: {repr(text[:100])}")

    print("\n--- Building the vocabulary ---")
    chars, char_to_id, id_to_char = build_vocab(text)
    vocab_size = len(chars)
    print(f"Unique characters ({vocab_size} total):\n  " + "".join(chars))

    print("\n--- Testing encode / decode ---")
    sample = "Hello, World!"
    encoded = encode(sample, char_to_id)
    decoded = decode(encoded, id_to_char)

    print(f"Original : {repr(sample)}")
    print(f"Encoded  : {encoded}")
    print(f"Decoded  : {repr(decoded)}")
    print(f"Round-trip matches: {sample == decoded}")

    print("\n--- Encoding the full dataset ---")
    all_ids = encode(text, char_to_id)

    # Store IDs in a PyTorch tensor for efficient model training
    data = torch.tensor(all_ids, dtype=torch.long)

    print(f"Full dataset as tensor: shape={data.shape}, dtype={data.dtype}")
    print(f"First 20 token IDs: {data[:20].tolist()}")
    print(f"Decoded back: {repr(decode(data[:20].tolist(), id_to_char))}")

    print("\n--- Splitting into train and validation sets ---")
    # Use 90% for training and 10% for validation to evaluate performance
    split = int(0.9 * len(data))
    train_data = data[:split]
    val_data   = data[split:]

    print(f"Training tokens  : {len(train_data):,}")
    print(f"Validation tokens: {len(val_data):,}")

    print("\nTokenizer ready! Ready for Chapter 5.")

---

### `src/examples/ch04_tokenizer_demo.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch04_tokenizer_demo.py"   # a cell has none, and the file uses it to find the text

"""Shows how different text inputs map to token IDs."""
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode

DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "shakespeare.txt")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chars, char_to_id, _ = build_vocab(text)

print("Demo: Encoding short business phrases")
phrases = [
    "invoice",
    "URGENT!",
    "Hello."
]

for p in phrases:
    # Notice how spaces and punctuation get their own IDs
    print(f"{repr(p):<15} -> {encode(p, char_to_id)}")